In [ ]:
# STEP 1: Import Libraries
# -----------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# STEP 2: Load Dataset
# -----------------------------------
df = pd.read_csv("learning style.csv")

print("Initial Shape:", df.shape)
print("First 5 rows:\n", df.head())

Initial Shape: (10000, 15)
First 5 rows:
   Student_ID  Age  Gender  Study_Hours_per_Week Preferred_Learning_Style  \
0     S00001   18  Female                    48              Kinesthetic   
1     S00002   29  Female                    30          Reading/Writing   
2     S00003   20  Female                    47              Kinesthetic   
3     S00004   23  Female                    13                 Auditory   
4     S00005   19  Female                    24                 Auditory   

   Online_Courses_Completed Participation_in_Discussions  \
0                        14                          Yes   
1                        20                           No   
2                        11                           No   
3                         0                          Yes   
4                        19                          Yes   

   Assignment_Completion_Rate (%)  Exam_Score (%)  Attendance_Rate (%)  \
0                             100              69                 

In [ ]:
# STEP 3: Data Preprocessing
# -----------------------------------

# Handle missing values
imputer = SimpleImputer(strategy="most_frequent")   # for categorical & numeric mix
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# Remove duplicates
df = df.drop_duplicates()

# Detect and handle outliers (IQR method for numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

In [ ]:
# Separate features (X) and target (y)
X = df.drop("Preferred_Learning_Style", axis=1)   # <-- replace column name with your target column
y = df["Preferred_Learning_Style"]


In [ ]:
# Encode target
le_target = LabelEncoder()
y = le_target.fit_transform(y)

# Encode categorical columns in X
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Now scale only numeric values
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# STEP 4: Train/Test Split
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 8000
Testing samples: 2000


In [ ]:
# STEP 5: Build Feedforward Neural Network (MLP)
# -----------------------------------
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y)), activation='softmax')   # output = number of classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# STEP 6: Train the Model
# -----------------------------------
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=300,   # you can increase (e.g. 100) for better accuracy
                    batch_size=32,
                    verbose=1)

Epoch 1/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2933 - loss: 1.3800 - val_accuracy: 0.2555 - val_loss: 1.3890
Epoch 2/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2800 - loss: 1.3793 - val_accuracy: 0.2590 - val_loss: 1.3887
Epoch 3/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2774 - loss: 1.3797 - val_accuracy: 0.2555 - val_loss: 1.3897
Epoch 4/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2880 - loss: 1.3788 - val_accuracy: 0.2485 - val_loss: 1.3912
Epoch 5/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2979 - loss: 1.3726 - val_accuracy: 0.2580 - val_loss: 1.3919
Epoch 6/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3079 - loss: 1.3692 - val_accuracy: 0.2510 - val_loss: 1.3894
Epoch 7/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2941 - loss: 1.3728 - val_accuracy: 0.2575 - val_loss: 1.3919
Epoch 8/300
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2988 - loss: 1.3730 - val_accu

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False  # freeze layers

x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


NameError: name 'num_classes' is not defined

In [ ]:
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 24.9 MB/s eta 0:00:00


In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import pandas as pd

# Load dataset
df = pd.read_csv("/content/preprocessed_learning_style.csv")
X = df.drop("Preferred_Learning_Style", axis=1)
y = df["Preferred_Learning_Style"]

# Encode categorical features
for col in X.columns:
    if X[col].dtype == 'object' or str(X[col].dtype) == 'category':
        X[col] = X[col].astype('category').cat.codes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    param = {
        "objective": "multi:softmax",
        "num_class": len(y.unique()),
        "eval_metric": "mlogloss",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("✅ Best Params:", study.best_params)
print("✅ Best Accuracy:", study.best_value)


[I 2025-09-02 03:44:18,766] A new study created in memory with name: no-name-a9e42213-9e50-4d7c-a7d0-01d60c5d6941
[I 2025-09-02 03:44:20,170] Trial 0 finished with value: 0.2475 and parameters: {'learning_rate': 0.15780456552146008, 'max_depth': 10, 'n_estimators': 106, 'subsample': 0.6282991054766133, 'colsample_bytree': 0.64477973608353}. Best is trial 0 with value: 0.2475.
[I 2025-09-02 03:44:21,417] Trial 1 finished with value: 0.2535 and parameters: {'learning_rate': 0.04789113786702833, 'max_depth': 4, 'n_estimators': 359, 'subsample': 0.8510001148230855, 'colsample_bytree': 0.6567226860483605}. Best is trial 1 with value: 0.2535.
[I 2025-09-02 03:44:23,200] Trial 2 finished with value: 0.252 and parameters: {'learning_rate': 0.020441812183025168, 'max_depth': 6, 'n_estimators': 298, 'subsample': 0.8799043341639452, 'colsample_bytree': 0.9410308315282374}. Best is trial 1 with value: 0.2535.
[I 2025-09-02 03:44:29,451] Trial 3 finished with value: 0.2565 and parameters: {'learnin

✅ Best Params: {'learning_rate': 0.18875797810247497, 'max_depth': 8, 'n_estimators': 195, 'subsample': 0.8692410142074761, 'colsample_bytree': 0.8663534662083662}
✅ Best Accuracy: 0.2665


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(y.unique()), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_split=0.2, verbose=1)


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2358 - loss: 1.4447 - val_accuracy: 0.2481 - val_loss: 1.3894
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2684 - loss: 1.3889 - val_accuracy: 0.2537 - val_loss: 1.3868
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2569 - loss: 1.3864 - val_accuracy: 0.2556 - val_loss: 1.3880
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2642 - loss: 1.3852 - val_accuracy: 0.2550 - val_loss: 1.3878
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2722 - loss: 1.3813 - val_accuracy: 0.2262 - val_loss: 1.3908
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2749 - loss: 1.3812 - val_accuracy: 0.2656 - val_loss: 1.3879
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2747 - loss: 1.3807 - val_accuracy: 0.2550 - val_loss: 1.3887
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2899 - loss: 1.3783 - val_accuracy: 0.2469 - val_

In [ ]:
# Install XGBoost if not already
!pip install xgboost scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
from tensorflow import keras
from tensorflow.keras import layers

# ===============================
# 1. Load Preprocessed Dataset
# ===============================
df = pd.read_csv("/content/preprocessed_learning_style.csv")

# Drop non-informative columns
if "Student_ID" in df.columns:
    df = df.drop("Student_ID", axis=1)

# ===============================
# 2. Define Features and Target
# ===============================
X = df.drop("Final_Grade", axis=1)
y = df["Final_Grade"]

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Encode target (A,B,C,D,F → numbers)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

from sklearn.metrics import classification_report

# Convert classes to string labels
class_names = [str(c) for c in le.classes_]

print("🎯 XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, target_names=class_names))

# ===============================
# 3. Model 1: XGBoost Classifier
# ===============================
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("🎯 XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


# ===============================
# 4. Model 2: Deep Neural Network
# ===============================
num_classes = len(np.unique(y_encoded))

dnn_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = dnn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

# Evaluate on test data
test_loss, test_acc = dnn_model.evaluate(X_test, y_test, verbose=0)
print("🤖 Neural Network Accuracy:", test_acc)

# ===============================
# 5. Compare & Save Best Model
# ===============================
if accuracy_score(y_test, y_pred_xgb) > test_acc:
    print("✅ Best Model: XGBoost")
else:
    print("✅ Best Model: Neural Network")


🎯 XGBoost Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       536
           1       1.00      1.00      1.00       491
           2       1.00      1.00      1.00       488
           3       1.00      1.00      1.00       485

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

🎯 XGBoost Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       536
           1       1.00      1.00      1.00       491
           2       1.00      1.00      1.00       488
           3       1.00      1.00      1.00       485

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5086 - loss: 1.0943 - val_accuracy: 0.8281 - val_loss: 0.4381
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7770 - loss: 0.4916 - val_accuracy: 0.8662 - val_loss: 0.2937
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8429 - loss: 0.3603 - val_accuracy: 0.9287 - val_loss: 0.2022
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8704 - loss: 0.2870 - val_accuracy: 0.9350 - val_loss: 0.1647
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9025 - loss: 0.2347 - val_accuracy: 0.9494 - val_loss: 0.1386
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9096 - loss: 0.2072 - val_accuracy: 0.9631 - val_loss: 0.1190
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9228 - loss: 0.1760 - val_accuracy: 0.9581 - val_loss: 0.1106
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9306 - loss: 0.1696 - val_accuracy: 0.9544 - val_

In [ ]:
import pickle
import joblib

# Save XGBoost model
joblib.dump(xgb_model, "xgb_final_grade_model.pkl")

# Save Neural Network model
dnn_model.save("dnn_final_grade_model.h5")

# Save preprocessing (LabelEncoder + feature columns)
with open("preprocessing_grade.pkl", "wb") as f:
    pickle.dump({
        "label_encoder": le,
        "feature_columns": X.columns.tolist()
    }, f)

print("✅ Models and preprocessing saved successfully!")


✅ Models and preprocessing saved successfully!


In [ ]:
import pandas as pd
import numpy as np
import pickle
import joblib
from tensorflow import keras

# ===============================
# Load Preprocessing & Models
# ===============================
with open("preprocessing_grade.pkl", "rb") as f:
    preprocess = pickle.load(f)

le = preprocess["label_encoder"]
feature_cols = preprocess["feature_columns"]

xgb_loaded = joblib.load("xgb_final_grade_model.pkl")
dnn_loaded = keras.models.load_model("dnn_final_grade_model.h5")


# ===============================
# Prediction Function
# ===============================
def predict_grade(user_input: dict, model_type="xgb"):
    # Convert dict → DataFrame
    user_df = pd.DataFrame([user_input])

    # One-hot encode & align with training features
    user_df = pd.get_dummies(user_df, drop_first=True)
    user_df = user_df.reindex(columns=feature_cols, fill_value=0)

    # Predict
    if model_type == "xgb":
        pred = xgb_loaded.predict(user_df)
    else:
        pred = np.argmax(dnn_loaded.predict(user_df), axis=1)

    return le.inverse_transform(pred)[0]


# ===============================
# User Input from Console
# ===============================
user_input = {
    "Hours_Studied": float(input("Enter hours studied: ")),
    "Attendance": float(input("Enter attendance percentage: ")),
    "Participation": input("Enter participation level (1–5): "),
    "Learning_Style": input("Enter learning style (e.g., Visual, Auditory, Reading/Writing, Kinesthetic): ")
}

# Predict using both models
print("\n🎯 Predicted Grade (XGBoost):", predict_grade(user_input, "xgb"))
print("🤖 Predicted Grade (DNN):", predict_grade(user_input, "dnn"))


Enter hours studied: 48
Enter attendance percentage: 66
Enter participation level (1–5): yes
Enter learning style (e.g., Visual, Auditory, Reading/Writing, Kinesthetic): 

🎯 Predicted Grade (XGBoost): 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 474ms/step
🤖 Predicted Grade (DNN): 2


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle

# Load dataset
df = pd.read_csv("learning style.csv")

# Drop Student_ID if exists
if "Student_ID" in df.columns:
    df = df.drop("Student_ID", axis=1)

# Define Features and Target
X = df.drop("Final_Grade", axis=1)
y = df["Final_Grade"]

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Save the feature columns
feature_columns = X.columns.tolist()

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save preprocessing objects
with open("preprocessing.pkl", "wb") as f:
    pickle.dump({"feature_columns": feature_columns, "label_encoder": le}, f)

print("✅ Saved preprocessing.pkl with feature columns and label encoder")


✅ Saved preprocessing.pkl with feature columns and label encoder


In [ ]:
import tensorflow as tf
import numpy as np

# Load trained model & preprocessing
model = tf.keras.models.load_model("dnn_final_grade_model.h5")
with open("preprocessing.pkl", "rb") as f:
    preprocess = pickle.load(f)

feature_columns = preprocess["feature_columns"]
le = preprocess["label_encoder"]

def predict_final_grade(user_input: dict):
    # Convert user input to DataFrame
    user_df = pd.DataFrame([user_input])

    # Apply same preprocessing (dummy encoding)
    user_df = pd.get_dummies(user_df, drop_first=True)

    # Reindex to match training features
    user_df = user_df.reindex(columns=feature_columns, fill_value=0)

    # Convert to numpy float array
    user_array = user_df.astype(float).to_numpy()

    # Predict
    pred_prob = model.predict(user_array)
    pred_class = np.argmax(pred_prob, axis=1)

    return le.inverse_transform(pred_class)[0]



In [ ]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

# ================================
# Load Dataset (for preprocessing reference)
# ================================
df = pd.read_csv("learning style.csv")

# Drop non-informative columns
if "Student_ID" in df.columns:
    df = df.drop("Student_ID", axis=1)

# Features & Target
X = df.drop("Final_Grade", axis=1)
y = df["Final_Grade"]

# One-hot encode categorical variables (Participation, Learning_Style, etc.)
X = pd.get_dummies(X, drop_first=True)

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save label encoder for later use
with open("grade_label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# ================================
# Load Trained Model
# ================================
model = tf.keras.models.load_model("dnn_final_grade_model.h5")

# ================================
# Prediction Function
# ================================
def predict_final_grade(user_input: dict):
    # Convert user input to DataFrame
    user_df = pd.DataFrame([user_input])

    # Apply same one-hot encoding as training
    user_df = pd.get_dummies(user_df)

    # Align with training columns
    user_df = user_df.reindex(columns=X.columns, fill_value=0)

    # Predict
    pred_prob = model.predict(user_df)
    pred_class = np.argmax(pred_prob, axis=1)

    # Decode back to original grade labels
    with open("grade_label_encoder.pkl", "rb") as f:
        le = pickle.load(f)

    return le.inverse_transform(pred_class)[0]

# ================================
# Example: User Input from Console
# ================================
user_input = {
    "Study_Hours": float(input("Enter Study Hours: ")),
    "Attendance": float(input("Enter Attendance %: ")),
    "Assignments_Submitted": int(input("Enter Assignments Submitted: ")),
    "Participation": input("Enter Participation (Yes/No): "),
    "Learning_Style": input("Enter Learning Style (Visual/Auditory/Kinesthetic): ")
}

print("\n📌 Predicted Final Grade:", predict_final_grade(user_input))


FileNotFoundError: [Errno 2] No such file or directory: 'learning style.csv'